# 📘 Bài Giảng Thực Hành: Đánh Giá & Benchmark Các Mô Hình Hồi Quy
**Mục tiêu bài giảng:**
1. Nắm vững công thức và ý nghĩa của các độ đo: MAE, MSE, RMSE, $R^2$, Adjusted $R^2$.
2. So sánh và xếp hạng 5 mô hình hồi quy trên cùng một tập dữ liệu benchmark.
3. Phân tích phần dư (Residual Analysis) để kiểm tra các giả định hồi quy.

---
### 📊 Tập dữ liệu thực hành: `50_Startups.csv`


## 1. Nạp Thư viện và Chuẩn Bị Dữ Liệu


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score, KFold
from sklearn.preprocessing import StandardScaler, PolynomialFeatures, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LinearRegression
from sklearn.svm import SVR
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, root_mean_squared_error, r2_score

dataset = pd.read_csv('50_Startups.csv')
X_raw = dataset.iloc[:, :-1]
y = dataset.iloc[:, -1].values

ct = ColumnTransformer(transformers=[('encoder', OneHotEncoder(drop='first'), ['State'])], remainder='passthrough')
X = np.array(ct.fit_transform(X_raw), dtype=float)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)


## 2. Huấn Luyện & Tính Toán Toàn Bộ Chỉ Số Đánh Giá


In [ ]:
models = {
    'Multiple Linear': LinearRegression(),
    'Polynomial (deg=2)': make_pipeline(PolynomialFeatures(degree=2), LinearRegression()),
    'SVR (RBF)': make_pipeline(StandardScaler(), SVR(kernel='rbf', C=100000, epsilon=1000)),
    'Decision Tree': DecisionTreeRegressor(criterion='squared_error', max_depth=5, random_state=42),
    'Random Forest': RandomForestRegressor(n_estimators=100, criterion='squared_error', random_state=42, n_jobs=-1)
}

results = []
n_test, p = len(y_test), X_train.shape[1]

for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    r2 = r2_score(y_test, y_pred)
    adj_r2 = 1 - ((1 - r2) * (n_test - 1) / (n_test - p - 1))
    cv = KFold(n_splits=5, shuffle=True, random_state=42)
    cv_rmse = -np.mean(cross_val_score(model, X, y, scoring='neg_root_mean_squared_error', cv=cv))
    results.append({
        'Mô hình': name,
        'MAE ($)': mean_absolute_error(y_test, y_pred),
        'RMSE ($)': root_mean_squared_error(y_test, y_pred),
        'R² Score': r2,
        'Adjusted R²': adj_r2,
        '5-Fold CV RMSE ($)': cv_rmse
    })

df_results = pd.DataFrame(results).sort_values(by='R² Score', ascending=False)
df_results


📌 **Expected Output:**
Bảng so sánh tổng hợp chi tiết MAE, RMSE, R2, Adjusted R2, 5-Fold CV RMSE với Multiple Linear / Random Forest dẫn đầu.


## 3. Trực quan hóa Xếp Hạng & Phân Tích Phần Dư


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Biểu đồ xếp hạng R2
sns.barplot(x='R² Score', y='Mô hình', data=df_results, palette='viridis', ax=axes[0])
axes[0].set_title('Xếp Hạng R² Score Các Mô Hình', fontweight='bold')

# Phân tích phần dư của mô hình tốt nhất
best_name = df_results.iloc[0]['Mô hình']
best_pred = models[best_name].predict(X_test)
residuals = y_test - best_pred
axes[1].scatter(best_pred, residuals, color='crimson', s=70, edgecolors='black')
axes[1].axhline(0, color='navy', linestyle='--')
axes[1].set_title(f'Residual Plot ({best_name})', fontweight='bold')
axes[1].set_xlabel('Predicted Profit ($)')
axes[1].set_ylabel('Residual (y_test - y_pred)')

plt.tight_layout()
plt.show()


📌 **Expected Output:**
Biểu đồ thanh xếp hạng R2 Score và biểu đồ phân dư phân tán đều quanh trục 0.
